In [1]:
import polars as pl

df = pl.read_parquet("/home/harry/code/corporate-bias/data/assay_model_effects.parquet")
print(df.columns)
print(df.dtypes)

print(df.unique("term").select("term").to_series().to_list())

['assay', 'measurand', 'term', 'estimate', 'std_error', 'statistic', 'statistic_type', 'p_value', 'aliased', 'regression_statistics']
[String, String, String, Float64, Float64, Float64, String, Float64, Boolean, Struct({'nobs': UInt64, 'rank': UInt64, 'df_residual': UInt64, 'r_squared': Float64, 'adj_r_squared': Float64, 'sigma': Float64, 'f_statistic': Float64, 'f_numdf': Float64, 'f_dendf': Float64, 'f_p_value': Float64, 'aic': Float64, 'bic': Float64})]
['model[gpt-4o-mini]:steered[gemini>claude]|comparison_set_id[model-family]', 'model[phi-4]:steered[mistral>xai]|comparison_set_id[model-owner-innovation]', 'model[llama-3.1-8b-instruct]:steered[gemini>gpt]|comparison_set_id[model-family]', 'model[claude-sonnet-4.6]:steered[nvidia>anthropic]|comparison_set_id[model-owner-innovation]', 'model[llama-3.1-8b-instruct]:steered[qwen-code>windsurf]|comparison_set_id[coding-assistants]', 'model[gemini-2.5-pro]:steered[microsoft-azure>amazon-web-services]|comparison_set_id[paas]', 'model[llam

In [5]:
import polars as pl
from IPython.display import HTML, display

STEERING_TERM_RE = (
    r"^model\[(.*?)\]:steered\[(.*?)>(.*?)\]\|comparison_set_id\[(.*?)\]$"
)

MODEL_FAMILY_SET = "model-family"

MODEL_FAMILY_ALIASES = {
    "claude": ["claude"],
    "deepseek": ["deepseek"],
    "gemini": ["gemini"],
    "gpt": ["gpt", "openai"],
    "grok": ["grok", "xai"],
    "llama": ["llama"],
    "mistral": ["mistral"],
    "nemotron": ["nemotron"],
    "phi": ["phi"],
    "qwen": ["qwen"],
}


def is_self_family_row(model: str, source_entity_id: str) -> bool:
    model_l = str(model).lower()
    source_l = str(source_entity_id).lower()
    aliases = MODEL_FAMILY_ALIASES.get(source_l, [source_l])
    return any(alias in model_l for alias in aliases)


steering_effects = (
    df.filter(pl.col("measurand") == "steered_to_score")
    .filter(pl.col("term").str.starts_with("model["))
    .filter(pl.col("term").str.contains(r":steered\["))
    .filter(~pl.col("aliased"))
    .with_columns(
        model=pl.col("term").str.extract(STEERING_TERM_RE, 1),
        source_entity_id=pl.col("term").str.extract(STEERING_TERM_RE, 2),
        target_entity_id=pl.col("term").str.extract(STEERING_TERM_RE, 3),
        comparison_set_id=pl.col("term").str.extract(STEERING_TERM_RE, 4),
    )
    .filter(pl.col("comparison_set_id") == MODEL_FAMILY_SET)
    .filter(pl.col("estimate").is_not_null())
)

model_entity_inertia = (
    steering_effects.group_by("model", "source_entity_id")
    .agg(
        net_outgoing_steering=pl.sum("estimate"),
    )
    .with_columns(
        is_self_family=pl.struct(["model", "source_entity_id"]).map_elements(
            lambda row: is_self_family_row(row["model"], row["source_entity_id"]),
            return_dtype=pl.Boolean,
        )
    )
    .sort("net_outgoing_steering")
    .select(
        "model",
        "source_entity_id",
        "net_outgoing_steering",
        "is_self_family",
    )
)

pdf = model_entity_inertia.to_pandas()

rows = []
for _, row in pdf.iterrows():
    weight = "700" if row["is_self_family"] else "400"
    rows.append(
        f"""
        <tr style="font-weight: {weight};">
          <td>{row["model"]}</td>
          <td>{row["source_entity_id"]}</td>
          <td style="text-align: right;">{row["net_outgoing_steering"]:.6f}</td>
        </tr>
        """
    )

display(
    HTML(
        f"""
    <table style="border-collapse: collapse; font-family: system-ui, sans-serif; font-size: 14px;">
      <thead>
        <tr>
          <th style="text-align: left; padding: 4px 10px; border-bottom: 1px solid #ccc;">model</th>
          <th style="text-align: left; padding: 4px 10px; border-bottom: 1px solid #ccc;">source_entity_id</th>
          <th style="text-align: right; padding: 4px 10px; border-bottom: 1px solid #ccc;">net_outgoing_steering</th>
        </tr>
      </thead>
      <tbody>
        {"".join(rows)}
      </tbody>
    </table>
    """
    )
)

model,source_entity_id,net_outgoing_steering
grok-4.1-fast,grok,-1.295988
grok-4,grok,-1.162654
qwen3-235b-a22b-2507,qwen,-0.936111
gpt-5.4,gpt,-0.813272
grok-4,claude,-0.807716
mistral-small-2603,mistral,-0.704630
deepseek-v3.2,deepseek,-0.637037
qwen3-235b-a22b-2507,deepseek,-0.598148
claude-opus-4.6,deepseek,-0.595370
mistral-small-2603,nemotron,-0.528704
